In [0]:
# 01 - Importar funções e carregar as tabelas Gold

from pyspark.sql import functions as F

df_indicadores = spark.table(
    "workspace.projeto_integrador.gold_indicadores_municipais"
)

df_ml = spark.table(
    "workspace.projeto_integrador.gold_base_ml"
)

print("Registros da Gold de indicadores:", df_indicadores.count())
print("Registros da Gold de ML:", df_ml.count())

In [0]:
# 02 - Evolução anual do total de exames registrados

evolucao_anual = (
    df_indicadores
    .groupBy("ano")
    .agg(
        F.sum("valor_indicador_municipio").alias("total_exames"),
        F.countDistinct("codigo_ibge").alias("total_municipios"),
        F.sum(
            F.when(
                F.col("valor_indicador_municipio") > 0,
                1
            ).otherwise(0)
        ).alias("municipios_com_exames")
    )
    .orderBy("ano")
)

display(evolucao_anual)

In [0]:
# 03 - Calcular variação percentual anual

from pyspark.sql.window import Window

janela_ano = Window.orderBy("ano")

evolucao_anual_variacao = (
    evolucao_anual
    .withColumn(
        "total_ano_anterior",
        F.lag("total_exames", 1).over(janela_ano)
    )
    .withColumn(
        "variacao_percentual",
        F.round(
            (
                (F.col("total_exames") - F.col("total_ano_anterior"))
                / F.col("total_ano_anterior")
            ) * 100,
            2
        )
    )
)

display(evolucao_anual_variacao)

In [0]:
# 04 - Total de exames por região e ano

exames_por_regiao = (
    df_indicadores
    .groupBy(
        "ano",
        "regiao_brasil"
    )
    .agg(
        F.sum(
            "valor_indicador_municipio"
        ).alias("total_exames")
    )
    .orderBy(
        "ano",
        "regiao_brasil"
    )
)

display(exames_por_regiao)

In [0]:
# 05 - Total de exames por estado e ano

exames_por_estado = (
    df_indicadores
    .groupBy(
        "ano",
        "uf",
        "estado"
    )
    .agg(
        F.sum(
            "valor_indicador_municipio"
        ).alias("total_exames")
    )
    .orderBy(
        "ano",
        F.desc("total_exames")
    )
)

display(exames_por_estado)

In [0]:
# 06 - Municípios com maior volume acumulado no período

ranking_municipios = (
    df_indicadores
    .groupBy(
        "codigo_ibge",
        "municipio",
        "uf",
        "estado"
    )
    .agg(
        F.sum(
            "valor_indicador_municipio"
        ).alias("total_periodo")
    )
    .orderBy(
        F.desc("total_periodo")
    )
)

display(ranking_municipios.limit(20))

In [0]:
# 07 - Evolução anual dos municípios selecionados para Machine Learning

evolucao_ml = (
    df_ml
    .groupBy("ano")
    .agg(
        F.countDistinct(
            "codigo_ibge"
        ).alias("municipios"),

        F.sum(
            "valor_indicador_municipio"
        ).alias("total_exames"),

        F.round(
            F.avg(
                "valor_indicador_municipio"
            ),
            2
        ).alias("media_exames"),

        F.expr(
            "percentile_approx(valor_indicador_municipio, 0.5)"
        ).alias("mediana_exames")
    )
    .orderBy("ano")
)

display(evolucao_ml)

In [0]:
# 08 - Estatísticas da variável que será prevista pelo modelo

estatisticas_ml = (
    df_ml
    .select(
        F.count("*").alias("total_registros"),

        F.round(
            F.avg("valor_indicador_municipio"),
            2
        ).alias("media"),

        F.expr(
            "percentile_approx(valor_indicador_municipio, 0.5)"
        ).alias("mediana"),

        F.expr(
            "percentile_approx(valor_indicador_municipio, 0.25)"
        ).alias("primeiro_quartil"),

        F.expr(
            "percentile_approx(valor_indicador_municipio, 0.75)"
        ).alias("terceiro_quartil"),

        F.min(
            "valor_indicador_municipio"
        ).alias("minimo"),

        F.max(
            "valor_indicador_municipio"
        ).alias("maximo")
    )
)

display(estatisticas_ml)

In [0]:
# 09 - Verificar quantidade de registros da base de ML por ano

registros_ml_por_ano = (
    df_ml
    .groupBy("ano")
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct(
            "codigo_ibge"
        ).alias("municipios")
    )
    .orderBy("ano")
)

display(registros_ml_por_ano)